# Day 42：高级 LangGraph 状态图（Checkpointer、Human-in-the-loop、并发节点、重试与 Writing Agent）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Day 42）！今天我们将实战演示 `SQLiteCheckpointer` 状态持久化与线程隔离、`interrupt_before` / `resume` 人工审批、`ParallelSectionRunner` 多线程并发章节撰写、`RetryPolicy` 节点级指数退避重试，以及完整的 **Writing Agent（大纲 -> 人工审批 -> 并发写章 -> 终稿润色）** 工作流（`v0.4.0-day42`）。

## 1. 环境初始化与导入 `project_iterations/day_42` 模块
定位仓库根目录并将 `project_iterations/day_42` 加入 `sys.path`。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day42_dir = repo_root / "project_iterations" / "day_42"
if str(day42_dir) not in sys.path:
    sys.path.insert(0, str(day42_dir))

from zhishu_agent import (
    ParallelSectionRunner,
    RetryPolicy,
    SQLiteCheckpointer,
    build_writing_agent_graph,
    execute_with_retry,
)

print("已加载模块路径:", day42_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_42


## 2. `SQLiteCheckpointer` 持久化与 `thread_id` 隔离
在临时目录创建 SQLite 检查点数据库，验证不同 `thread_id` 的状态严格隔离，且重新实例化后可从磁盘恢复。

In [2]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / "demo_ckpt.db"
    cp1 = SQLiteCheckpointer(db_path)
    cp1.save_checkpoint("tenant-a", step=1, next_node="human_approval", state={"topic": "RAG 2.0"}, status="interrupted")
    cp1.save_checkpoint("tenant-b", step=1, next_node="__END__", state={"topic": "多智能体路由"}, status="completed")

    cp2 = SQLiteCheckpointer(db_path)
    print("已持久化线程列表:", cp2.list_threads())
    print("tenant-a 最新状态:", cp2.get_latest("tenant-a").state)
    print("tenant-b 最新状态:", cp2.get_latest("tenant-b").state)

已持久化线程列表: ['tenant-a', 'tenant-b']
tenant-a 最新状态: {'topic': 'RAG 2.0'}
tenant-b 最新状态: {'topic': '多智能体路由'}


## 3. `ParallelSectionRunner` 并发写章与 `RetryPolicy` 节点级重试
模拟第 2 章首次调用瞬时超时，观察 `RetryPolicy` 自动重试恢复，并验证 `ThreadPoolExecutor` 并发执行后仍保持原始章节顺序。

In [3]:
import threading

attempts = {}
lock = threading.Lock()

def draft_section(idx: int, title: str) -> str:
    with lock:
        attempts[idx] = attempts.get(idx, 0) + 1
        cur = attempts[idx]
    if idx == 1 and cur == 1:
        raise TimeoutError("模拟第 2 节模型接口瞬时超时")
    return f"[{idx + 1}] {title} -> 撰写成功 (attempts={cur})"

runner = ParallelSectionRunner(
    max_workers=3,
    retry_policy=RetryPolicy(max_retries=2, initial_delay_s=0.005, retry_on=(TimeoutError,)),
)
results = runner.run(["背景与挑战", "核心架构设计", "生产治理与观测"], draft_section)
for r in results:
    print(r.content)

[1] 背景与挑战 -> 撰写成功 (attempts=1)
[2] 核心架构设计 -> 撰写成功 (attempts=2)
[3] 生产治理与观测 -> 撰写成功 (attempts=1)


## 4. 完整 Writing Agent：`Outline -> Human Approval -> Parallel Chapters -> Final Polish`
启动写作任务后在 `human_approval` 前自动暂停，人工通过 `resume(thread_id, updated_state)` 修改大纲并批准，驱动并发写章与终稿润色。

In [4]:
with tempfile.TemporaryDirectory() as tmpdir:
    cp = SQLiteCheckpointer(Path(tmpdir) / "writing_workflow.db")
    app = build_writing_agent_graph(
        checkpointer=cp,
        max_workers=4,
        node_retry_policy=RetryPolicy(max_retries=2, initial_delay_s=0.005),
    )

    paused = app.invoke({"topic": "企业级 LangGraph 编排实践"}, thread_id="nb-write-01")
    print("阶段1状态:", paused.status, "| 暂停节点:", paused.interrupted_at)
    print("初始大纲:", paused.state["outline"])

    edited_outline = list(paused.state["outline"]) + ["第四章：检查点持久化与人工审批最佳实践"]
    completed = app.resume(
        "nb-write-01",
        updated_state={
            "approved": True,
            "outline": edited_outline,
            "human_feedback": "大纲已补充持久化章节，准予并发撰写",
        },
    )
    print("阶段2状态:", completed.status, "| 轨迹:", completed.execution_trace)
    print("\n--- 终稿输出 ---\n" + completed.state["final_article"])

阶段1状态: interrupted | 暂停节点: human_approval
初始大纲: ['第一章：企业级 LangGraph 编排实践 的背景与核心痛点', '第二章：企业级 LangGraph 编排实践 的系统架构与关键技术选型', '第三章：企业级 LangGraph 编排实践 的工程落地与未来演进']
阶段2状态: completed | 轨迹: ['human_approval(attempts=1)', 'parallel_chapters(attempts=1)', 'final_polish(attempts=1)']

--- 终稿输出 ---
# 企业级 LangGraph 编排实践
> 审稿批注：大纲已补充持久化章节，准予并发撰写

### 第一章：企业级 LangGraph 编排实践 的背景与核心痛点
本节围绕“企业级 LangGraph 编排实践”第 1 部分展开：结合企业级智能体平台实践，明确状态持久化、人工审批门禁与并发执行在生产稳定性中的核心价值。

### 第二章：企业级 LangGraph 编排实践 的系统架构与关键技术选型
本节围绕“企业级 LangGraph 编排实践”第 2 部分展开：结合企业级智能体平台实践，明确状态持久化、人工审批门禁与并发执行在生产稳定性中的核心价值。

### 第三章：企业级 LangGraph 编排实践 的工程落地与未来演进
本节围绕“企业级 LangGraph 编排实践”第 3 部分展开：结合企业级智能体平台实践，明确状态持久化、人工审批门禁与并发执行在生产稳定性中的核心价值。

### 第四章：检查点持久化与人工审批最佳实践
本节围绕“企业级 LangGraph 编排实践”第 4 部分展开：结合企业级智能体平台实践，明确状态持久化、人工审批门禁与并发执行在生产稳定性中的核心价值。

---
共完成 4 个章节的并发撰写与终稿润色。
